In [3]:
# Instalación de dependencias
!pip install transformers accelerate datasets -q
!pip install huggingface_hub -q

# Autenticación con Hugging Face
from huggingface_hub import login
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from dotenv import load_dotenv
from huggingface_hub import login
from datetime import datetime
from pathlib import Path
from tqdm.notebook import tqdm
from google.colab import drive
from google.colab import files

#
import torch
import os
import re
import time

In [ ]:
# cargamos el token de huggingface
drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/.env')
token = os.getenv("HF_TOKEN")
login(token=token)

# vamos a utilizar gemma
model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
PROMPT_STYLES = [

# 1. Intuitivo pero sin ejemplo al principio
"""
You are helping a beginner understand a computer vision concept.

QUESTION:
{question}

Requirements:
- start with a clear intuitive explanation, but do NOT start with an example, scenario, or analogy,
- then provide a precise definition,
- if useful, include one short example only near the end,
- expand acronyms the first time they appear,
- do not start with "In computer vision", "This concept", "Imagine", "Suppose", or "For example",
- no code, no bullet points.

Write two short paragraphs.
""".strip(),

# 2. Técnico
"""
You are answering a university-level exam question about computer vision.

QUESTION:
{question}

Requirements:
- start with a precise definition,
- explain the underlying mechanism in a technical way,
- include one limitation or drawback,
- expand acronyms the first time they appear,
- avoid generic openings,
- do not start with "Imagine", "Suppose", or "For example",
- no code or bullet points.

Write one compact paragraph.
""".strip(),

# 3. Comparación
"""
You are explaining a concept by comparing it with a related idea.

QUESTION:
{question}

Requirements:
- define the concept,
- compare it with a closely related technique or concept,
- highlight at least one key difference,
- mention when it is preferred,
- expand acronyms the first time they appear,
- avoid starting with an example or scenario,
- no code, no bullet points.

Write two short paragraphs.
""".strip(),

# 4. Basado en problema, pero sin narrativa de ejemplo al inicio
"""
You are explaining a concept through the problem it solves.

QUESTION:
{question}

Requirements:
- start by stating the problem or challenge in a direct academic way,
- explain how the concept addresses it,
- include one practical application only at the end if useful,
- expand acronyms the first time they appear,
- avoid narrative openings such as "Imagine", "Suppose", or "Consider a situation",
- no code or bullet points.

Write one or two paragraphs.
""".strip(),

# 5. Error común
"""
You are teaching a concept and addressing common misunderstandings.

QUESTION:
{question}

Requirements:
- explain the concept clearly,
- include one common mistake or misconception,
- correct that misunderstanding,
- include one application if useful,
- expand acronyms the first time they appear,
- avoid starting with an example or scenario,
- no code, no bullet points.

Write two short paragraphs.
""".strip(),

# 6. Antes era ejemplo primero; ahora ejemplo opcional al final
"""
You are explaining a concept in a didactic way.

QUESTION:
{question}

Requirements:
- begin with a direct explanation of the concept,
- then explain the idea behind it,
- include a brief definition,
- if useful, add one short example only in the second paragraph,
- expand acronyms the first time they appear,
- avoid generic sentence openings and do not start with "Imagine", "Suppose", or "For example",
- no code or bullet points.

Write two short paragraphs.
""".strip(),

# 7. Paso a paso narrativo
"""
You are explaining a concept in a structured but narrative way.

QUESTION:
{question}

Requirements:
- explain the concept step by step in plain text (no bullet points),
- describe how it works internally,
- include one application near the end,
- expand acronyms the first time they appear,
- do not begin with a scenario or hypothetical example,
- no code.

Write two short paragraphs.
""".strip(),

# 8. Contexto práctico pero con definición primero
"""
You are explaining how a concept is used in real applications.

QUESTION:
{question}

Requirements:
- begin with a brief and direct definition,
- then focus on how it is used in practice,
- include one real-world application,
- mention one limitation,
- expand acronyms the first time they appear,
- avoid starting with an anecdotal example or "Imagine",
- no code or bullet points.

Write one or two paragraphs.
""".strip(),

# 9. Breve + ampliación
"""
You are explaining a concept concisely and then expanding it.

QUESTION:
{question}

Requirements:
- start with a very short definition,
- then expand with a deeper explanation,
- include one application only if it adds value,
- expand acronyms the first time they appear,
- avoid repeating sentence structures,
- do not start with "Imagine", "Suppose", or "For example",
- no code.

Write two short paragraphs.
""".strip(),

# 10. Enfoque conceptual puro
"""
You are explaining the core idea behind a concept.

QUESTION:
{question}

Requirements:
- focus on the main idea and purpose,
- avoid superficial or generic explanations,
- mention one use case only at the end if helpful,
- expand acronyms the first time they appear,
- do not begin with an example, scenario, or analogy,
- no code or bullet points.

Write one or two paragraphs.
""".strip(),

]

In [ ]:
# Definimos un nº de tokens junto a una temperatura baja y valor penalti de repetición para mejorar la calidad del promtp
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.55
TOP_P = 0.9
REPETITION_PENALTY = 1.1
BATCH_SIZE = 8

# Cargamos el txt con las preguntas
uploaded = files.upload()
concepts_file_t = list(uploaded.keys())[0]

# funcion auxiliar para leer las preguntas de un txt
def load_questions_from_txt(file_path):
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el txt: {file_path}")

    questions = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                questions.append(line)
    return questions


def build_prompt_theory(question, idx):
    res = PROMPT_STYLES[idx % len(PROMPT_STYLES)]
    return res.format(question=question)

# Limpiamos el texto
def clean_text_txt(text):
    if "=== ANSWER START ===" in text:
        text = text.split("=== ANSWER START ===", 1)[-1]
    if "=== ANSWER END ===" in text:
        text = text.split("=== ANSWER END ===", 1)[0]
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\*\*", "", text)
    return text.strip()


def normalize(text):
    text = text.replace("\n\n", "\n")
    text = text.replace("\n", "\\n")
    return text.strip()


def generate_answers_batch(prompts):
    text_prompts = []

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text_prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False
        )
        text_prompts.append(text_prompt)

    inputs = tokenizer(
        text_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    answers = []
    for i, length in enumerate(prompt_lengths):
        generated_ids = output_ids[i][length:]
        raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        answers.append(clean_text_txt(raw_text))

    return answers


# Ejecutamos el generador
start_time = time.time()

questions = load_questions_from_txt(concepts_file_t)
qa_results = []

for start_idx in tqdm(
    range(0, len(questions), BATCH_SIZE),
    desc="Generation in progress",
    unit="batch",
    dynamic_ncols=True,
    leave=True
):
    batch_questions = questions[start_idx:start_idx + BATCH_SIZE]
    batch_prompts = [
        build_prompt_theory(question, start_idx + j)
        for j, question in enumerate(batch_questions)
    ]

    batch_answers = generate_answers_batch(batch_prompts)

    for question, answer in zip(batch_questions, batch_answers):
        qa_results.append({
            "question": question,
            "answer": answer
        })

# Guardamos los resultados generados en un nuevo archivo con estructura Q + A + Salto de linea
elapsed_time = time.time() - start_time
elapsed_str = f"{elapsed_time:.1f}segundos"

timestamp = datetime.now().strftime("%Y-%m-%d--%H-%M")
filename = f"{timestamp}-{elapsed_str}-qa_computervision_THEORY.txt"

with open(filename, "w", encoding="utf-8") as f:
    for qa in qa_results:
        q = normalize(qa["question"])
        a = normalize(qa["answer"])
        f.write(q + "\n")
        f.write(a + "\n\n")

print(f"\n==Archivo guardado en==: {filename}")
print(f"Se han procesado {len(qa_results)} preguntas")
print(f"El tiempo total de ejecución ha sido de : {elapsed_str}")
files.download(str(filename))